# 01. Semantic Search of SEC Filings

This is the first of two notebooks introducing **retrieval-augmented
generation (RAG)**. RAG is a technique for grounding large language models
in source documents so they don't have to rely on memorized (and often
stale) knowledge. Every RAG system has two stages:

1. **Retrieval** -- find the most relevant documents for a given query.
2. **Generation** -- feed those documents to an LLM and ask it to answer.

This notebook covers stage 1: building and querying a semantic search
index over SEC filings. The next notebook (02) covers stage 2, where we
measure how much retrieved context actually improves LLM accuracy.

## Why search SEC filings?

On August 7, 2018, Elon Musk tweeted:
> "Am considering taking Tesla private at $420. Funding secured."

This triggered an SEC investigation and a \$40 million settlement. A
researcher might ask: what did Tesla's prior SEC filings say about
going-private transactions? About funding arrangements? Were these
disclosures filed before or after the tweet -- and what does that imply
about what management knew?

The problem is scale. SEC filings are long (10-Ks can exceed 200 pages),
filled with legal boilerplate, and there are thousands of them. A keyword
search for "risk" returns virtually every filing. We need a search system
that understands **meaning**, not just keywords.

## The data

Our corpus comes from the **WRDS SEC Analytics Suite**, which provides
cleaned full-text versions of EDGAR filings. The build pipeline (run via
`doit`) downloads filings for a sample of well-known companies, extracts
target sections (MD&A, Risk Factors, etc.), chunks them into passages,
and encodes each passage as a 384-dimensional vector using the
`all-MiniLM-L6-v2` sentence transformer model.

## Semantic embeddings and FAISS

Instead of matching keywords, we convert text into **vectors** that
capture meaning. Texts with similar meaning produce similar vectors,
regardless of exact word choice:

```
"Tesla considers going private"     -> [0.23, -0.15, 0.87, ...]  (384 dims)
"Company evaluates privatization"   -> [0.21, -0.18, 0.85, ...]  (similar!)
"The weather is sunny today"        -> [-0.45, 0.72, -0.11, ...]  (different)
```

**FAISS** (Facebook AI Similarity Search) is a library from Meta AI
Research for efficient similarity search over large collections of
vectors. Given a query vector, FAISS finds the k nearest neighbors in
the index -- the passages whose meaning is closest to the query. For
small corpora (< 100K vectors) we use exact inner-product search
(`IndexFlatIP`); for larger corpora, approximate methods like IVF or
HNSW trade a small amount of accuracy for orders-of-magnitude speedup.

This notebook loads a pre-built FAISS index and performs
**event-conditioned search**: given an event date and a natural-language
query, it retrieves the most semantically relevant filings filed before
and after the event.

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import faiss
import torch
torch.set_num_threads(1)
from sentence_transformers import SentenceTransformer
from typing import List, Dict
from pathlib import Path
from datetime import datetime, date
from collections import Counter

In [2]:
from settings import config
DATA_DIR = config("DATA_DIR")
INDEX_FILENAME = "sec_filings.faiss"
META_FILENAME = "sec_filings_meta.json"

## Helper functions

The functions below handle loading the FAISS index, querying it, and
filtering results by event date and company. They are defined up front
so the rest of the notebook reads as a straight narrative.

In [3]:
def load_index(output_dir: str):
    """Load FAISS index and metadata."""
    index_path = os.path.join(output_dir, INDEX_FILENAME)
    meta_path = os.path.join(output_dir, META_FILENAME)

    print(f"Loading index from {index_path}")
    index = faiss.read_index(index_path)

    print(f"Loading metadata from {meta_path}")
    with open(meta_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return index, data["texts"], data["metadatas"]


def get_filing_date_range(metadatas):
    dates = []

    for m in metadatas:
        d = m.get("filing_date")
        if d is None:
            continue
        if isinstance(d, str):
            d = datetime.fromisoformat(d).date()
        dates.append(d)

    if not dates:
        raise ValueError("No filing dates found in metadata.")

    return min(dates), max(dates)


def search(
    query: str, model, index, texts: List[str], metadatas: List[Dict], k: int = 5
):
    """Search the FAISS index."""
    q_emb = model.encode(
        [query],
        convert_to_numpy=True,
        device="cpu",
        normalize_embeddings=True,
    ).astype("float32")

    D, I = index.search(q_emb, k)

    results = []
    for score, idx in zip(D[0], I[0]):
        results.append(
            {
                "score": float(score),
                "text": texts[idx],
                "meta": metadatas[idx],
            }
        )
    return results


def search_event(
    query: str,
    event_date,
    model,
    index,
    texts,
    metadatas,
    company: str | None = None,
    strict_company: bool = False,
    k_pre: int = 10,
    k_post: int = 10,
    window_days: int = 90,
    oversample_factor: int = 10,
    diagnostics: bool = True,
):
    """
    Event-conditioned semantic search with coverage diagnostics.
    """

    # -------------------------------
    # Embed query
    # -------------------------------
    q_emb = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    D, I = index.search(q_emb, (k_pre + k_post) * oversample_factor)

    pre_results, post_results = [], []
    seen = set()

    # -------------------------------
    # Diagnostics counters
    # -------------------------------
    diag = Counter()
    diag_company_matches = Counter()

    for score, idx in zip(D[0], I[0]):
        meta = metadatas[idx]

        # Filing date
        filing_date = meta.get("filing_date")
        if filing_date is None:
            continue
        if isinstance(filing_date, str):
            filing_date = datetime.fromisoformat(filing_date).date()

        delta = (filing_date - event_date).days
        if abs(delta) > window_days:
            continue

        diag["in_window"] += 1

        # Company diagnostics
        meta_company = meta.get("company_name")
        if company is not None:
            if company_match(company, meta_company):
                diag["company_match"] += 1
                diag_company_matches[meta_company] += 1
            else:
                diag["company_mismatch"] += 1

        # Company filtering (only if strict)
        company_ok = True
        if company is not None:
            company_ok = company_match(company, meta_company)
            if strict_company and not company_ok:
                continue

        # Dedup
        key = (meta["accession"], meta["section"])
        if key in seen:
            continue
        seen.add(key)

        record = {
            "score": float(score),
            "text": texts[idx],
            "meta": meta,
            "delta_days": delta,
            "company_match": company_ok,
        }

        if delta < 0 and len(pre_results) < k_pre:
            pre_results.append(record)
        elif delta >= 0 and len(post_results) < k_post:
            post_results.append(record)

        if len(pre_results) >= k_pre and len(post_results) >= k_post:
            break

    # Ranking
    def rank_key(r):
        return (r["company_match"], r["score"])

    pre_results.sort(key=rank_key, reverse=True)
    post_results.sort(key=rank_key, reverse=True)

    # -------------------------------
    # Coverage diagnostics output
    # -------------------------------
    if diagnostics and company is not None:
        print("\n" + "=" * 80)
        print("COMPANY COVERAGE DIAGNOSTICS")
        print("=" * 80)
        print(f"Company focus: {company}")
        print(f"Event window: +/-{window_days} days")
        print(f"Candidates in window: {diag['in_window']}")
        print(f"Company-matching candidates: {diag['company_match']}")
        print(f"Company-mismatching candidates: {diag['company_mismatch']}")

        if diag_company_matches:
            print("\nMatched company names seen in metadata:")
            for name, cnt in diag_company_matches.most_common(5):
                print(f"  {name}: {cnt}")
        else:
            print("\n  No filings in window matched the company name.")

        if diag["company_match"] == 0:
            print("\nLIKELY CAUSE:")
            print("- No Microsoft filings exist in the +/-window, OR")
            print("- XBRL company names differ materially from 'Microsoft'")
            print("- Try widening window_days or inspecting raw XBRL company names")

    return pre_results, post_results


def normalize_company(name: str) -> str:
    if not name:
        return ""

    name = name.lower()
    name = name.replace("&", "and")

    for suffix in [
        " incorporated",
        " inc",
        " inc.",
        " corporation",
        " corp",
        " corp.",
        " ltd",
        " llc",
        " plc",
        " co",
    ]:
        name = name.replace(suffix, "")

    return " ".join(name.split())


def company_match(query_company: str, meta_company: str) -> bool:
    """
    Returns True if company names are 'close enough'.
    """
    if not query_company or not meta_company:
        return True  # permissive fallback

    q = normalize_company(query_company)
    m = normalize_company(meta_company)

    # direct containment
    if q in m or m in q:
        return True

    # token overlap (handles 'johnson johnson' vs 'johnson and johnson')
    q_tokens = set(q.split())
    m_tokens = set(m.split())

    overlap = q_tokens & m_tokens
    return len(overlap) >= max(1, len(q_tokens) - 1)


def print_disclosures(disclosure_type, label):
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    for i, r in enumerate(disclosure_type, 1):
        meta = r["meta"]
        print(f"\n{i}. Score: {r['score']:.4f}")
        print(f"Company: {meta.get('company_name')}")
        print(f"Form: {meta['filing_type']}")
        print(
            f"Filed: {meta['filing_date']} (T{'+' if int(r['delta_days']) > 0 else ''}{r['delta_days']})"
        )
        print(f"Section: {meta['section']}")
        print(f"EDGAR Detail URL: {meta['edgar_detail_url']}")
        print(r["text"][:1200])
        print("\n" + "-" * 150)

## Load the pre-built index

The build pipeline (`doit`) has already chunked our SEC filings, encoded
them with the sentence transformer, and saved the resulting FAISS index
to disk. Here we load that index along with the passage texts and
metadata (company name, filing type, filing date, EDGAR URL, etc.).
We also load the same sentence transformer model so we can encode
queries at search time.

In [4]:
INDEX_DIR = os.path.join(DATA_DIR, "faiss_index", "small")

if not os.path.exists(os.path.join(INDEX_DIR, INDEX_FILENAME)):
    raise FileNotFoundError(
        f"FAISS index not found at {INDEX_DIR}. "
        "Run 'doit' or 'python src/build_faiss_index.py' first."
    )

index, texts, metadatas = load_index(INDEX_DIR)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Loaded index with {index.ntotal} vectors")

Loading index from /Users/jbejarano/GitRepositories/finm-32900-ALL/case_study_sec_filings/_data/faiss_index/small/sec_filings.faiss
Loading metadata from /Users/jbejarano/GitRepositories/finm-32900-ALL/case_study_sec_filings/_data/faiss_index/small/sec_filings_meta.json


Loaded index with 603750 vectors


## Event-driven search

Plain semantic search returns the most relevant passages regardless of
when they were filed. But for event studies we often care about
**timing**: what did the company disclose *before* an event, and how did
disclosures change *after*?

The `search_event` function adds a date dimension. Given an event date,
a time window, and an optional company filter, it retrieves the top-k
most relevant passages filed before the event and the top-k filed after.
This lets us compare pre- and post-event language side by side.

In [5]:
event_date = date(2024, 12, 24)

min_date, max_date = get_filing_date_range(metadatas)

print("\n" + "=" * 80)
print("FILING DATE COVERAGE CHECK")
print("=" * 80)
print(f"Available filings date range: {min_date} -> {max_date}")
print(f"Requested event date:        {event_date}")

if not (min_date <= event_date <= max_date):
    print("\n WARNING: Event date is OUTSIDE the available filings range.")
    print("Event-conditioned search will return no results.")
else:
    print("\n Event date is within the available filings range.")


FILING DATE COVERAGE CHECK
Available filings date range: 2014-01-23 -> 2026-02-11
Requested event date:        2024-12-24

 Event date is within the available filings range.


In [6]:
top_k = 10

print("\n" + "=" * 80)
print("EVENT-CONDITIONED SEARCH RESULTS")
print("=" * 80)

company_focus = "Microsoft"  # None

query = """
FTC antitrust investigations, pending litigation, legal settlements
"""

set(
    m["company_name"]
    for m in metadatas
    if company_focus is not None
    and company_focus.lower() in (m.get("company_name") or "").lower()
)

pre, post = search_event(
    query=query,
    event_date=event_date,
    model=model,
    index=index,
    texts=texts,
    metadatas=metadatas,
    company=company_focus,
    k_pre=top_k,
    k_post=top_k,
    window_days=180,
)

print_disclosures(post, "POST-EVENT DISCLOSURES")
print_disclosures(pre, "PRE-EVENT DISCLOSURES")


EVENT-CONDITIONED SEARCH RESULTS



COMPANY COVERAGE DIAGNOSTICS
Company focus: Microsoft
Event window: +/-180 days
Candidates in window: 23
Company-matching candidates: 0
Company-mismatching candidates: 23

  No filings in window matched the company name.

LIKELY CAUSE:
- No Microsoft filings exist in the +/-window, OR
- XBRL company names differ materially from 'Microsoft'
- Try widening window_days or inspecting raw XBRL company names

POST-EVENT DISCLOSURES

1. Score: 0.5698
Company: Meta Platforms, Inc.
Form: 10-K
Filed: 2025-01-30 (T+37)
Section: In 2024, we did not identify any cybersecurity threats that have materially affected or are reasonably likely to materially affect our business strategy, results of operations, or financial condition. However, despite our efforts, we cannot eliminate all risks from cybersecurity threats, or provide assurances that we have not experienced undetected cybersecurity incidents. For additional information about these risks, see Part I, Item 1A, "Risk Factors" in this Annual Rep

## Simple search

For comparison, here is a plain semantic search with no date filtering.
This is useful when you want the best matches across the entire corpus
regardless of timing.

In [7]:
# search query
query = "johnson and johnson restructuring losses or expenses"
top_k = 25

print("\n" + "=" * 50)

results = search(query, model, index, texts, metadatas, k=top_k)

for i, r in enumerate(results, 1):
    meta = r["meta"]
    print(f"\nResult {i} - {r['score']:.2%}")
    print(f"File: {meta['filename']}")
    print(f"EDGAR detail URL: {meta.get('edgar_detail_url', 'N/A')}")
    # Short preview; adjust length as needed
    print(f"Text preview:\n...")
    print(r["text"][:2000])



Result 1 - 63.28%
File: 0001193125-16-742796.txt
EDGAR detail URL: https://www.sec.gov/Archives/edgar/data/789019/000119312516742796/0001193125-16-742796-index.html
Text preview:
...
10-Q | contingencies for these issues within the next 12 months. We also

Three Months Ended September 30, 2016 2015 Dividends and interest
income $ 
293 $ 
199 Interest expense (437 
) (249 
) Net recognized gains on
investments 405 2 Net losses on derivatives (94 
) (103 
) Net losses on foreign currency
remeasurements (52 
) (36 
) Other (15 
) (93 
) Total $ 
100 $ 
(280 
) Changes
in the restructuring liability were as follows: (In millions) Severance Other 
(a) Total Restructuring liability as of
June 30, 2016 $ 
470 $ 
239 $ 
709 Restructuring charges 0 0 0 Cash paid (161 
) (15 
) (176 
) Restructuring liability as of
September 30, 2016 $ 
309 $ 
224 $ 
533 (a) Other primarily reflects activities associated
with the consolidation of our facilities and manufacturing
operations, including contract 